In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
from birddog.database import (
    Database,
)
from birddog.nocodb_database import (
    clone_table_schema,
    copy_records,
    rename_field,
    copy_formula_field,
    list_formula_fields,
    list_lookup_fields,
    create_lookup_field,
)

2026-08-17 15:46:41,544 [INFO] Using local nocodb api: http://localhost:8080


In [3]:
local_db = Database()
local_db._host

2026-08-17 15:46:42,437 [INFO] creating NocoDBDatabase(host=http://localhost:8080, base_id=p79fvr9cjqgpv5n) instance
2026-08-17 15:46:42,454 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                          5.00    62.93     7.00       0.00            4


'http://localhost:8080'

In [4]:
nocodb_aws_host = os.environ["BIRDDOG_AWS_NOCODB_HOST"]
nocodb_aws_token = os.environ["BIRDDOG_AWS_NOCODB_API_TOKEN"]
nocodb_aws_base_id = os.environ["BIRDDOG_AWS_BASE_ID"]

In [5]:
aws_db = Database(host=nocodb_aws_host, api_token=nocodb_aws_token, base_id=nocodb_aws_base_id)
aws_db._host

2026-08-17 15:46:46,116 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance


'http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com'

In [6]:
nocodb_cloud_host = os.environ["BIRDDOG_CLOUD_NOCODB_HOST"]
nocodb_cloud_token = os.environ["BIRDDOG_CLOUD_NOCODB_API_TOKEN"]
nocodb_cloud_base_id = os.environ["BIRDDOG_CLOUD_BASE_ID"]

In [7]:
cloud_db = Database(host=nocodb_cloud_host, api_token=nocodb_cloud_token, base_id=nocodb_cloud_base_id)

2026-08-17 15:46:51,073 [INFO] creating NocoDBDatabase(host=https://app.nocodb.com, base_id=p2vtf1kcfs0gl2x) instance


In [8]:
def _delete_all(db, table_name):
    db.delete(table_name, db.get_all_ids(table_name))
    
def clone_db(src_db, dest_db):
    try:
        clone_table_schema(src_db, "Schema", dest_db, "Schema")
    except ValueError as err:
        print("Schema table already exists")
    try:    
        clone_table_schema(src_db, "Schema Values", dest_db, "Schema Values")
    except ValueError as err:
        print("Schema Values table already exists")

    _delete_all(dest_db, "Schema")
    copy_records(src_db, "Schema", dest_db, "Schema")

    _delete_all(dest_db, "Schema Values")
    copy_records(src_db, "Schema Values", dest_db, "Schema Values")

    dest_db.load_schema()
    clone_table_schema(src_db, "Pages", dest_db, "Pages")
    clone_table_schema(src_db, "Documents", dest_db, "Documents")
    dest_db.load_schema()

In [ ]:
clone_db(aws_db, cloud_db)

In [ ]:
rename_field(local_db, local_db._field_id("Pages", "Pages"), "parent")
rename_field(local_db, local_db._field_id("Documents", "Pages"), "owning_pages")
local_db.load_schema()

In [ ]:
list_lookup_fields(aws_db, "Documents")

In [ ]:
def list_rollup_fields(db, table):
    info = db._get_table_info(table)
    return [c["title"] for c in info["columns"] if c["uidt"] == "Rollup"]

In [ ]:
rename_field(local_db, local_db._field_id("Documents", "Pages"), "owning_pages")

In [ ]:
rename_field(aws_db, "c9bgrdnnxmr90et", "duplicate_of")

In [ ]:
rename_field(cloud_db, cloud_db._field_id("Pages", "Pages"), "parent")
#rename_field(local_db, local_db._field_id("Documents", "Pages"), "owning_pages")


In [ ]:
rename_field(cloud_db, cloud_db._field_id("Documents", "Documents"), "duplicate_of")


In [ ]:
cloud_db._field_id("Pages", "parent")

In [ ]:
import time

In [ ]:
s = time.time()
i=aws_db.get_all_ids("Documents")
print(len(i), time.time() - s)

In [ ]:
recs = aws_db.scan_all("Schema Values")
ids = local_db.write("Schema Values", recs)

In [ ]:
rename_field(local_db, "cs9mptgh0b6s0jp", "duplicate_of")

In [9]:
_delete_all(cloud_db, "Documents")

2026-08-17 15:47:42,555 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                          5.00     0.03     7.00       0.00            4
  nocodb.cloud:api                       3.50     1.55     0.02       0.00            4
  nocodb.internal:api                   20.00     0.05    39.00       0.00           24
2026-08-17 15:48:42,699 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                          5.00     0.00     7.00       0.00            4
  nocodb.cloud:api                       4.25     2.43     3.00       0.00            4
  nocodb.internal:api                   20.00     0.00    39.00       0.00           2

In [10]:
_delete_all(cloud_db, "Pages")

2026-08-17 15:51:20,635 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                          5.00     0.00     7.00       0.00            4
  nocodb.cloud:api                       4.50     0.37     3.00       0.00            4
  nocodb.internal:api                   20.00     0.00    39.00       0.00           24
2026-08-17 15:52:20,826 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                          5.00     0.00     7.00       0.00            4
  nocodb.cloud:api                       4.50     3.17     3.00       0.00            4
  nocodb.internal:api                   20.00     0.00    39.00       0.00           2